In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(), #here we scale the values b/w 0 to 1 and convert the images into pytorch tensor values
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #normalise the scaled images b/w -1 to 1 (1st bracket = S.D, 2nd bracket = Mean)
])
trainset = CIFAR10(root="./data", train=True, download=False, transform=transform) #we need the training data
testset = CIFAR10(root="./data", train=False, download=False, transform=transform)#we need the testing data

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True) #form batches after shuffling under DataLoader
testloader = DataLoader(testset, batch_size=64)

Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),#Convolutional Layer 1
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),#Conv Layer 2
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),#Conv layer 3
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
#Max pooling halves the height and width with every convolutional layer and the 0/p channel increases by the formula n-f+2p/s + 1
        #32-->16-->8-->4
        self.fc_layers = nn.Sequential( #Fully Connected Layer
            nn.Linear(4*4*128, 256), #4*4*128 features after flattening
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x): #for Frwd Propagation
        x = self.conv_layers(x) #x is the the 4*4*128 which is flattened b4 sending it into the sequential nn 
        x = x.view(x.size(0), -1) # flattening in pytorch
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN() #object created for the CNN model

In [6]:
criterion = nn.CrossEntropyLoss() #for multi class classification (alrdy has softmax activation function implemented)
optimizer = optim.Adam(model.parameters()) #Optimise the model

###Training our CNN

In [ ]:
epochs = 10# no of times the full model will be visualised by the neural network

for epoch in range(epochs):
    epoch_training_loss = 0.0 #training loss of each epoch

    for images, labels in trainloader:
        optimizer.zero_grad()#not pilling up the gradiants and reseting to 0 for every optimisation step of each epoch
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # Back prop
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

In [8]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.8
